# Possession Play — Spielerdatenbank bauen (Kaggle)

Dieses Notebook erzeugt das vollständige `PLAYERS`-Array für das Spiel aus dem
Transfermarkt-Dataset **`davidcariboo/player-scores`**. Es führt die Logik der
beiden Repo-Skripte `data-pipeline/build_db.py` und `data-pipeline/make_game_json.py`
nacheinander aus.

## So benutzt du es

1. Auf https://www.kaggle.com → **Create → New Notebook**.
2. Rechts **Add Input → Datasets** → nach **`davidcariboo/player-scores`** suchen
   und hinzufügen. Es mountet dann unter `/kaggle/input/player-scores/`.
3. Den Inhalt dieses Notebooks einfügen (bzw. dieses `.ipynb` hochladen).
4. Oben **Run All**.
5. Den **Vereins-Prüfbericht** in der Ausgabe kontrollieren (jeder Spiel-Verein
   sollte mind. einen TM-Namen gematcht haben; `⚠️ KEIN TREFFER` heißt: den
   Teilstring in `GAME_CLUBS` anpassen und erneut laufen lassen).
6. Rechts unter **Output → /kaggle/working/** die Datei **`players_game.js`**
   herunterladen.

## Danach (im Repo)

Der Inhalt von `players_game.js` ersetzt das `export const PLAYERS = [ … ];` in
**`src/players.js`** — siehe `data-pipeline/README.md`. Das Notebook schreibt die
Ausgabe bereits mit `export const`-Präfix, sodass du den kompletten Dateiinhalt
1:1 nach `src/players.js` übernehmen kannst.

> Hinweise aus den Skripten: Einsatz-Daten reichen i.d.R. nur ~2012 zurück;
> für ältere Vereinszugehörigkeiten müsste Transferhistorie ergänzt werden.
> "seit 2000" = `games.season >= 2000`.

In [ ]:
# ── Konfiguration & Helfer ─────────────────────────────────────────────
# Logik aus build_db.py + make_game_json.py. Einzige Anpassung ggü. den
# lokalen Skripten: Pfade zeigen auf den Kaggle-Mount bzw. /kaggle/working.
from pathlib import Path
import json, unicodedata
import pandas as pd

DATA = Path("/kaggle/input/player-scores")   # statt ./data
OUT  = Path("/kaggle/working")               # statt ./out
OUT.mkdir(parents=True, exist_ok=True)

# Transfermarkt-Wettbewerbs-IDs der fünf Topligen (build_db.py)
TOP5 = {
    "GB1": ("Premier League", "England"),
    "ES1": ("LaLiga",         "Spain"),
    "IT1": ("Serie A",        "Italy"),
    "L1":  ("Bundesliga",     "Germany"),
    "FR1": ("Ligue 1",        "France"),
}
SINCE = 2000

# Spielkey -> Teilstring, der im TM-Vereinsnamen vorkommt (make_game_json.py)
GAME_CLUBS = {
    "FCB": "bayern", "BVB": "dortmund", "RBL": "leipzig", "B04": "leverkusen",
    "SGE": "frankfurt", "BMG": "gladbach", "VFB": "stuttgart", "WOB": "wolfsburg", "SVW": "bremen",
    "MCI": "manchester city", "MUN": "manchester united", "LIV": "liverpool", "CHE": "chelsea",
    "ARS": "arsenal", "TOT": "tottenham", "NEW": "newcastle", "EVE": "everton", "AVL": "aston villa",
    "BAR": "barcelona", "RMA": "real madrid", "ATM": "atletico", "SEV": "sevilla",
    "VAL": "valencia", "VIL": "villarreal",
    "JUV": "juventus", "MIL": "ac milan", "INT": "inter", "NAP": "napoli", "ROM": "roma", "LAZ": "lazio",
    "PSG": "paris", "ASM": "monaco", "OM": "marseille", "OL": "lyon", "LIL": "lille",
    "POR": "porto", "SLB": "benfica", "SCP": "sporting cp",
    "AJA": "ajax", "PSV": "psv", "FEY": "feyenoord",
}

NATION_MAP = {
    "france": "FRA", "germany": "GER", "spain": "ESP", "italy": "ITA", "netherlands": "NED",
    "belgium": "BEL", "croatia": "CRO", "england": "ENG", "portugal": "PRT", "japan": "JPN",
    "brazil": "BRA", "argentina": "ARG", "mexico": "MEX", "nigeria": "NGA",
    "cote d'ivoire": "CIV", "ivory coast": "CIV", "senegal": "SEN", "colombia": "COL",
    "united states": "USA", "usa": "USA",
}

# True = nur Spieler mit >=1 Spiel-Verein (kompaktes JSON, empfohlen)
ONLY_GAME_CLUBS = True

def norm(s: str) -> str:
    s = unicodedata.normalize("NFD", str(s)).encode("ascii", "ignore").decode()
    return s.lower().strip()

def club_key_for(tm_name: str):
    n = norm(tm_name)
    for key, needle in GAME_CLUBS.items():
        if needle in n:
            return key
    return None

print("Input:", DATA, "->", sorted(p.name for p in DATA.glob("*.csv"))[:12])

In [ ]:
# ── Schritt 1: build_db.py (Topligen seit 2000 filtern) ────────────────
players      = pd.read_csv(DATA / "players.csv")
clubs        = pd.read_csv(DATA / "clubs.csv")
games        = pd.read_csv(DATA / "games.csv")
appearances  = pd.read_csv(DATA / "appearances.csv")

# 1) Spiele der Topligen seit 2000 -> Saison je game_id
g = games[games["competition_id"].isin(TOP5) & (games["season"] >= SINCE)]
g = g[["game_id", "competition_id", "season"]]

# 2) Einsätze in genau diesen Spielen
app = appearances.merge(g, on="game_id", how="inner")

# 4) players.csv – nur Spieler mit >=1 Topliga-Einsatz seit 2000
pids = app["player_id"].unique()
pl = players[players["player_id"].isin(pids)].copy()
pl = pl[["player_id", "name", "first_name", "last_name",
         "date_of_birth", "position", "country_of_citizenship"]]
pl.columns = ["tm_player_id", "name", "first_name", "last_name",
              "date_of_birth", "position", "citizenship"]
# Mononyme (Rodri, Vinícius): wenn last_name leer, vollen Namen nehmen
pl["last_name"] = pl["last_name"].fillna("").where(
    pl["last_name"].notna() & (pl["last_name"] != ""), pl["name"])
pl.to_csv(OUT / "players.csv", index=False)

# 5) clubs.csv – Vereine, die in diesen Einsätzen vorkommen
cids = app["player_club_id"].unique()
cl = clubs[clubs["club_id"].isin(cids)].copy()
cl = cl[["club_id", "name", "domestic_competition_id"]]
cl.columns = ["tm_club_id", "name", "tm_competition_id"]
cl.to_csv(OUT / "clubs.csv", index=False)

# 6) player_club_spells.csv – (Spieler, Verein) -> Saison-Spanne
spells = (
    app.groupby(["player_id", "player_club_id"])["season"]
       .agg(["min", "max"])
       .reset_index()
)
spells.columns = ["tm_player_id", "tm_club_id", "season_start", "season_end"]
spells.to_csv(OUT / "player_club_spells.csv", index=False)

print(f"OK  players={len(pl):>6}  clubs={len(cl):>4}  spells={len(spells):>6}  -> {OUT}/")

In [ ]:
# ── Schritt 2: make_game_json.py (auf Spiel-Vereine mappen) ────────────
players = pd.read_csv(OUT / "players.csv")
clubs   = pd.read_csv(OUT / "clubs.csv")
spells  = pd.read_csv(OUT / "player_club_spells.csv")

# tm_club_id -> Spielkey (nur für gemappte Vereine)
club_key = {}
matched_names = {}  # Spielkey -> Menge der TM-Namen (Prüfbericht)
for _, c in clubs.iterrows():
    k = club_key_for(c["name"])
    if k:
        club_key[c["tm_club_id"]] = k
        matched_names.setdefault(k, set()).add(c["name"])

# PRÜFBERICHT: je Spielkey die zugeordneten TM-Vereinsnamen kontrollieren.
print("── Vereins-Zuordnung (kontrollieren!) ──")
for k in GAME_CLUBS:
    names = sorted(matched_names.get(k, []))
    flag = "" if names else "  ⚠️ KEIN TREFFER"
    print(f"  {k}: {', '.join(names) if names else '—'}{flag}")
print("────────────────────────────────────────")

# Spieler -> Set der Spielkeys
pclubs = {}
for _, s in spells.iterrows():
    k = club_key.get(s["tm_club_id"])
    if k:
        pclubs.setdefault(s["tm_player_id"], set()).add(k)

out = []
for _, p in players.iterrows():
    keys = sorted(pclubs.get(p["tm_player_id"], []))
    if ONLY_GAME_CLUBS and not keys:
        continue  # für dieses Spiel irrelevant
    nat = NATION_MAP.get(norm(p.get("citizenship", "")))
    try:
        by = int(str(p["date_of_birth"])[:4])
    except Exception:
        continue
    out.append({
        "n": p["name"],
        "ln": p["last_name"] if isinstance(p["last_name"], str) and p["last_name"].strip() else p["name"],
        "by": by,
        "nat": [nat] if nat else [],
        "clubs": keys,
    })

out.sort(key=lambda r: r["n"])
body = ",\n  ".join(json.dumps(r, ensure_ascii=False) for r in out)
# export const, damit der Inhalt 1:1 nach src/players.js übernommen werden kann
js = "export const PLAYERS = [\n  " + body + "\n];\n"
(OUT / "players_game.js").write_text(js, encoding="utf-8")
print(f"OK  {len(out)} Spieler -> {OUT}/players_game.js")
print("Download: rechts unter Output, dann Inhalt nach src/players.js übernehmen.")